# 02 — استخراج اسلات‌ها (Slot Filling) با مدل زبانی

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("."))
from chatbot_common import call_llm, extract_json, load_schema, load_raw_examples, bio_slots_to_dict, VAL_DIR

SCHEMA = load_schema()
print(f"schema برای {len(SCHEMA)} intent بارگذاری شد.")


## ۱) استخراج همه‌ی اسلات‌های یک intent از یک پیام

In [ ]:
def build_multi_slot_system_prompt(intent: str) -> str:
    slot_defs = SCHEMA.get(intent, [])
    if not slot_defs:
        return None
    lines = [
        "شما یک استخراج‌کننده‌ی اطلاعات ساختاریافته برای یک چت‌بات بانکی فارسی هستید.",
        f'کاربر قصد "{intent}" را دارد. اسلات‌های مرتبط با این قصد و توضیح هرکدام:',
    ]
    for s in slot_defs:
        lines.append(f"- {s['slot']}: {s['question']}")
    lines += [
        "",
        "متن کاربر را بخوانید و فقط اسلات‌هایی را که واقعاً در متن ذکر شده‌اند استخراج کنید.",
        "اگر اسلاتی در متن نبود، آن را در خروجی نیاورید (حدس نزنید).",
        "فقط یک JSON با این فرمت برگردانید، بدون هیچ توضیح یا Markdown اضافه:",
        '{"slots": {"<نام اسلات>": "<مقدار>", ...}}',
    ]
    return "\n".join(lines)


def extract_all_slots(intent: str, text: str) -> dict:
    system_prompt = build_multi_slot_system_prompt(intent)
    if system_prompt is None:
        return {}
    raw = call_llm(system_prompt, text, max_tokens=500, temperature=0.0)
    try:
        parsed = extract_json(raw)
        return parsed.get("slots", {})
    except Exception:
        return {}


## ۲) استخراج/نرمال‌سازی یک اسلات از پاسخ کاربر به یک سؤال مشخص

In [ ]:
def build_single_slot_system_prompt(slot_name: str, question: str) -> str:
    return (
        "شما یک استخراج‌کننده‌ی مقدار برای یک فیلد مشخص در یک فرم بانکی فارسی هستید.\n"
        f"سؤالی که از کاربر پرسیده شده: «{question}»\n"
        f"نام فیلد: {slot_name}\n"
        "پاسخ خام کاربر را بخوانید و فقط مقدار تمیز و نرمال‌شده را استخراج کنید "
        "(مثلاً بله/خیر را به true/false تبدیل کنید، اعداد فارسی را به انگلیسی، اعداد را بدون کاما بنویسید).\n"
        "اگر پاسخ کاربر معتبر/مرتبط نبود یا اطلاعات کافی نداشت، مقدار را null بگذارید.\n"
        "فقط یک JSON با این فرمت برگردانید:\n"
        '{"value": "<مقدار تمیزشده>", "valid": true}'
    )


def extract_single_slot(slot_name: str, question: str, user_reply: str) -> dict:
    """برمی‌گرداند {'value': ..., 'valid': bool}"""
    system_prompt = build_single_slot_system_prompt(slot_name, question)
    raw = call_llm(system_prompt, user_reply, max_tokens=200, temperature=0.0)
    try:
        parsed = extract_json(raw)
        value = parsed.get("value")
        valid = bool(parsed.get("valid", value is not None))
        if value in (None, "null", ""):
            return {"value": None, "valid": False}
        return {"value": value, "valid": valid}
    except Exception:
        # اگر مدل JSON برنگرداند، حداقل پاسخ خام کاربر را به‌عنوان مقدار نگه می‌داریم
        return {"value": user_reply.strip(), "valid": True}


## تست سریع

In [ ]:
# استخراج چندگانه: یک پیام که چند اسلات را همزمان شامل می‌شود
demo_intent = "card2card"
demo_text = "می‌خوام ۲۰۰ هزار تومان بابت بدهی به شماره کارت 6037991234567890 انتقال بدم"
print("چندگانه:", extract_all_slots(demo_intent, demo_text))

# استخراج تکی: پاسخ به یک سؤال مشخص
print("تکی:", extract_single_slot("activate_ib", "آیا قصد فعال‌سازی اینترنت بانک را دارید؟", "بله حتماً"))
print("تکی:", extract_single_slot("starter_amount", "مقدار اولیه چقدر است؟", "یک میلیون و پانصد هزار تومان"))


## ارزیابی 


In [ ]:
from tqdm import tqdm
import pandas as pd

SAMPLE_SIZE = 50
raw_val = load_raw_examples(VAL_DIR)

if raw_val:
    intent_code_to_name = {}
    if os.path.exists(os.path.join("data", "Intent_and_Slot_Mapping_Table-2.csv")):
        mdf = pd.read_csv(os.path.join("data", "Intent_and_Slot_Mapping_Table-2.csv"))
        intent_code_to_name = dict(zip(mdf["code"], mdf["intent"]))

    examples = raw_val[:SAMPLE_SIZE]
    tp = fp = fn = 0
    for ex in tqdm(examples):
        intent_name = intent_code_to_name.get(ex.get("intent_id"), ex.get("intent_id"))
        gold = bio_slots_to_dict(ex["input_text"], ex.get("slots", []))
        pred = extract_all_slots(intent_name, ex["input_text"])

        gold_keys, pred_keys = set(gold.keys()), set(pred.keys())
        tp += len(gold_keys & pred_keys)
        fp += len(pred_keys - gold_keys)
        fn += len(gold_keys - pred_keys)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    print(f"Slot-name Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")
else:
    print("داده‌ای  پیدا نشد .")
